# Teacher-Student Distillation Notebook

This notebook runs experiment `12` using a teacher-student distillation anomaly detector with a configurable teacher backbone on the shared `64x64` wafer split.

Updated workflow:
- load the shared teacher-student config
- inspect the dataset split and CUDA status
- optionally train the model and save artifacts under `artifacts/`
- run the default shared evaluation for apples-to-apples comparison
- optionally launch a same-notebook ablation sweep over teacher layers and `topk_ratio`
- run a post-training teacher-student score sweep over branch weights and wafer-level reductions
- compare the default score against stronger rescoring variants in the same notebook


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/teacher_student/resnet18/x64/main/train_config.toml`
- Artifact root: `experiments/anomaly_detection/teacher_student/resnet18/x64/main/artifacts/ts_resnet18`
- Notes:
  - This notebook now defaults to its experiment-local ResNet18 teacher config.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean


In [ ]:
CONFIG_NAME = 'resnet18_teacher_main'
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet18/x64/main/train_config.toml'
THRESHOLD_QUANTILE = 0.95
RETRAIN = False
RUN_ABLATION_SWEEP = False
ABLATION_PROFILE = 'focused'
if ABLATION_PROFILE == 'focused':
    ABLATION_LAYERS = ['layer1', 'layer2']
    ABLATION_TOPK_RATIOS = [0.15, 0.2, 0.25]
    ABLATION_EPOCHS_OVERRIDE = 15
else:
    ABLATION_LAYERS = ['layer1', 'layer2', 'layer3']
    ABLATION_TOPK_RATIOS = [0.1, 0.15, 0.2, 0.25, 0.3]
    ABLATION_EPOCHS_OVERRIDE = 20
ABLATION_REUSE_EXISTING = True
SCORE_SWEEP_WEIGHTS = [(1.0, 1.0), (1.0, 0.0), (0.0, 1.0), (2.0, 1.0), (1.0, 2.0), (1.0, 0.5), (0.5, 1.0)]
SCORE_SWEEP_REDUCTIONS = [('mean', None), ('max', None), ('topk_mean', 0.01), ('topk_mean', 0.05), ('topk_mean', 0.1), ('topk_mean', 0.2)]
config = load_toml(CONFIG_PATH)
output_dir = REPO_ROOT / config['run']['output_dir']
evaluation_dir = output_dir / 'results' / 'evaluation'
evaluation_dir.mkdir(parents=True, exist_ok=True)
plots_dir = output_dir / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
print(f'Using config: {CONFIG_NAME} -> {CONFIG_PATH}')
config


In [ ]:
requested_device = str(config['training'].get('device', 'auto'))
if requested_device == 'auto':
    resolved_device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
else:
    resolved_device_name = requested_device
device = torch.device(resolved_device_name)
cuda_summary = {'config_device': requested_device, 'torch_cuda_available': torch.cuda.is_available(), 'resolved_device': resolved_device_name, 'cuda_device_count': torch.cuda.device_count()}
if torch.cuda.is_available():
    cuda_summary['cuda_device_name'] = torch.cuda.get_device_name(0)
cuda_summary


In [ ]:
image_size = int(config['data'].get('image_size', 64))
batch_size = int(config['data'].get('batch_size', 64))
num_workers = int(config['data'].get('num_workers', 0))
metadata_path = REPO_ROOT / config['data']['metadata_csv']
metadata = pd.read_csv(metadata_path)
display(metadata.head())
display(metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index())
train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')


In [ ]:
if RETRAIN:
    train_command = [sys.executable, '-u', str(REPO_ROOT / 'scripts' / 'train_ts_distillation.py'), '--config', str(CONFIG_PATH)]
    print('Running:', ' '.join(train_command))
    process = subprocess.Popen(train_command, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, train_command)
else:
    print(f'Skipping training. Using existing checkpoint at {best_model_path}')
    if not best_model_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {best_model_path}.')


In [ ]:
history_df = pd.read_json(output_dir / 'results' / 'history.json')
training_summary = json.loads((output_dir / 'results' / 'summary.json').read_text(encoding='utf-8'))
display(history_df.tail())
training_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val')
axes[0].set_title('Teacher-Student Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['train_distillation_loss'], marker='o', label='train distillation')
axes[1].plot(history_df['epoch'], history_df['val_distillation_loss'], marker='o', label='val distillation')
axes[1].plot(history_df['epoch'], history_df['train_autoencoder_loss'], marker='o', linestyle='--', label='train feature AE')
axes[1].plot(history_df['epoch'], history_df['val_autoencoder_loss'], marker='o', linestyle='--', label='val feature AE')
axes[1].set_title('Teacher-Student Branch Losses')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
training_curves_path = plots_dir / 'training_curves.png'
plt.tight_layout()
plt.savefig(training_curves_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved training curves to {training_curves_path}')


In [ ]:
if RETRAIN:
    evaluate_command = [sys.executable, str(REPO_ROOT / 'scripts' / 'evaluate_reconstruction_model.py'), '--checkpoint', str(best_model_path), '--config', str(CONFIG_PATH), '--model-type', 'ts_distillation', '--threshold-quantile', str(THRESHOLD_QUANTILE), '--output-dir', str(evaluation_dir)]
    print('Running:', ' '.join(evaluate_command))
    subprocess.run(evaluate_command, cwd=REPO_ROOT, check=True)
default_summary_path = evaluation_dir / 'summary.json'
if not default_summary_path.exists():
    raise FileNotFoundError(f'Evaluation summary not found: {default_summary_path}')
default_eval_summary = json.loads(default_summary_path.read_text(encoding='utf-8'))
display(pd.Series(default_eval_summary['metrics_at_validation_threshold']))
display(pd.Series(default_eval_summary['best_threshold_sweep']))


## Optional TS Ablation Sweep

This section can run a compact retraining sweep over teacher feature layers and deployment `topk_ratio` values from the same notebook. The default `focused` preset tests only the highest-value variants first, while `full` restores the broader grid. It writes temporary configs, trains each variant with the standard script, evaluates it with the shared evaluator, and saves a sortable summary table.


In [ ]:
from copy import deepcopy

def format_toml_value(value):
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, int) and (not isinstance(value, bool)):
        return str(value)
    if isinstance(value, float):
        return repr(value)
    return json.dumps(str(value))

def write_simple_toml(path, config_dict):
    lines = []
    for section_name, section_values in config_dict.items():
        lines.append(f'[{section_name}]')
        for key, value in section_values.items():
            lines.append(f'{key} = {format_toml_value(value)}')
        lines.append('')
    path.write_text('\n'.join(lines).rstrip() + '\n', encoding='utf-8')

def stream_command(command, cwd):
    print('Running:', ' '.join((str(part) for part in command)))
    process = subprocess.Popen([str(part) for part in command], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def build_ablation_variants(base_config, layers, topk_ratios, epochs_override=None):
    variants = []
    backbone_name = str(base_config['model']['teacher_backbone']).lower()
    image_size_name = f"x{int(base_config['data'].get('image_size', 64))}"
    for teacher_layer in layers:
        for topk_ratio in topk_ratios:
            variant_config = deepcopy(base_config)
            variant_name = f'ts_{backbone_name}_{teacher_layer}_topk{int(round(topk_ratio * 100)):02d}'
            variant_config['run']['output_dir'] = f'experiments/anomaly_detection/teacher_student/resnet18/x64/main/artifacts/ablation_variants/{variant_name}'
            variant_config['model']['teacher_layer'] = teacher_layer
            variant_config['model']['topk_ratio'] = float(topk_ratio)
            variant_config['model']['score_student_weight'] = 1.0
            variant_config['model']['score_autoencoder_weight'] = 0.0
            variant_config['training']['resume_from'] = ''
            if epochs_override is not None:
                variant_config['training']['epochs'] = int(epochs_override)
            variants.append({'name': variant_name, 'teacher_layer': teacher_layer, 'topk_ratio': float(topk_ratio), 'config': variant_config})
    return variants

def run_ablation_variant(variant, threshold_quantile, reuse_existing):
    variant_name = variant['name']
    variant_config = variant['config']
    generated_config_dir = output_dir.parent / 'generated_configs'
    generated_config_dir.mkdir(parents=True, exist_ok=True)
    config_path = generated_config_dir / f'{variant_name}.toml'
    write_simple_toml(config_path, variant_config)
    variant_output_dir = REPO_ROOT / variant_config['run']['output_dir']
    variant_output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = variant_output_dir / 'checkpoints' / 'best_model.pt'
    evaluation_output_dir = variant_output_dir / 'results' / 'evaluation'
    evaluation_output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = evaluation_output_dir / 'summary.json'
    if reuse_existing and checkpoint_path.exists():
        print(f'Reusing checkpoint for {variant_name}: {checkpoint_path}')
    else:
        stream_command([sys.executable, '-u', REPO_ROOT / 'scripts' / 'train_ts_distillation.py', '--config', config_path], cwd=REPO_ROOT)
    if reuse_existing and summary_path.exists():
        print(f'Reusing evaluation for {variant_name}: {summary_path}')
    else:
        stream_command([sys.executable, REPO_ROOT / 'scripts' / 'evaluate_reconstruction_model.py', '--checkpoint', checkpoint_path, '--config', config_path, '--model-type', 'ts_distillation', '--threshold-quantile', str(threshold_quantile), '--output-dir', evaluation_output_dir], cwd=REPO_ROOT)
    train_summary = json.loads((variant_output_dir / 'results' / 'summary.json').read_text(encoding='utf-8'))
    evaluation_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    metrics = evaluation_summary['metrics_at_validation_threshold']
    best_sweep = evaluation_summary['best_threshold_sweep']
    return {'name': variant_name, 'teacher_backbone': variant_config['model']['teacher_backbone'], 'teacher_layer': variant['teacher_layer'], 'topk_ratio': variant['topk_ratio'], 'epochs': int(variant_config['training']['epochs']), 'best_epoch': int(train_summary['best_epoch']), 'best_val_loss': float(train_summary['best_val_loss']), 'precision': float(metrics['precision']), 'recall': float(metrics['recall']), 'f1': float(metrics['f1']), 'auroc': float(metrics['auroc']), 'auprc': float(metrics['auprc']), 'best_sweep_f1': float(best_sweep['f1']), 'threshold': float(metrics['threshold']), 'output_dir': str(variant_output_dir), 'config_path': str(config_path)}
ablation_variants = build_ablation_variants(base_config=config, layers=ABLATION_LAYERS, topk_ratios=ABLATION_TOPK_RATIOS, epochs_override=ABLATION_EPOCHS_OVERRIDE)
pd.DataFrame([{'name': variant['name'], 'teacher_layer': variant['teacher_layer'], 'topk_ratio': variant['topk_ratio'], 'epochs': variant['config']['training']['epochs']} for variant in ablation_variants])


In [ ]:
ablation_summary_path = evaluation_dir / 'ablation_sweep_summary.csv'
if RUN_ABLATION_SWEEP:
    ablation_rows = []
    for variant in ablation_variants:
        print(f"\n=== Ablation: {variant['name']} ===")
        ablation_rows.append(run_ablation_variant(variant=variant, threshold_quantile=THRESHOLD_QUANTILE, reuse_existing=ABLATION_REUSE_EXISTING))
    ablation_df = pd.DataFrame(ablation_rows).sort_values(['f1', 'auroc'], ascending=False).reset_index(drop=True)
    ablation_df.to_csv(ablation_summary_path, index=False)
    print(f'Saved ablation sweep summary to {ablation_summary_path}')
elif ablation_summary_path.exists():
    ablation_df = pd.read_csv(ablation_summary_path)
    print(f'Loaded existing ablation summary from {ablation_summary_path}')
else:
    ablation_df = pd.DataFrame()
    print('Skipping ablation sweep.')
ablation_df.head(10)


In [ ]:
if ablation_df.empty:
    print('No ablation sweep results available yet.')
else:
    display(ablation_df)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    top_rows = ablation_df.head(8)
    axes[0].bar(top_rows['name'], top_rows['f1'], color='#2a9d8f')
    axes[0].set_title('Top Ablations by F1')
    axes[0].set_ylabel('F1')
    axes[0].tick_params(axis='x', rotation=45)
    axes[1].bar(top_rows['name'], top_rows['auroc'], color='#457b9d')
    axes[1].set_title('Top Ablations by AUROC')
    axes[1].set_ylabel('AUROC')
    axes[1].tick_params(axis='x', rotation=45)
    ablation_plot_path = plots_dir / 'ablation_summary.png'
    plt.tight_layout()
    plt.savefig(ablation_plot_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved ablation summary plot to {ablation_plot_path}')


## Teacher-Student Score Sweep

The first teacher-student result looked more like a calibration problem than a training-collapse problem. This sweep keeps the trained checkpoint fixed and tests whether different branch weights and wafer-level reductions produce a stronger operating point under the same validation-threshold rule.


In [ ]:
checkpoint = torch.load(best_model_path, map_location='cpu')
score_sweep_model = build_ts_distillation_from_config(config, image_size=image_size)
score_sweep_model.load_state_dict(checkpoint['model_state_dict'])
score_sweep_model.to(device)
score_sweep_model.eval()

def collect_normalized_maps(model, dataloader, device):
    student_maps = []
    auto_maps = []
    labels = []
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            student_map, auto_map = model.raw_anomaly_maps(inputs)
            student_maps.append((student_map / model.student_map_scale.clamp_min(1e-06)).cpu())
            auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-06)).cpu())
            labels.append(batch_labels.cpu())
    return (torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy())
val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(score_sweep_model, val_loader, device)
test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(score_sweep_model, test_loader, device)
print(tuple(val_student_maps.shape), tuple(test_student_maps.shape))


In [ ]:
def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
    if reduction == 'mean':
        return spatial_mean(anomaly_map)
    if reduction == 'max':
        return spatial_max(anomaly_map)
    return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)
score_sweep_rows = []
for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
    val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
    test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
    for reduction, topk_ratio in SCORE_SWEEP_REDUCTIONS:
        val_scores = reduce_anomaly_map(val_map, reduction, topk_ratio).numpy()
        test_scores = reduce_anomaly_map(test_map, reduction, topk_ratio).numpy()
        threshold = float(pd.Series(val_scores[val_labels == 0]).quantile(THRESHOLD_QUANTILE))
        metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
        _, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
        score_sweep_rows.append({'name': f's{student_weight:g}_a{auto_weight:g}_{reduction}' + ('' if topk_ratio is None else f'_r{topk_ratio:.2f}'), 'student_weight': student_weight, 'auto_weight': auto_weight, 'reduction': reduction, 'topk_ratio': np.nan if topk_ratio is None else topk_ratio, 'threshold': threshold, 'precision': metrics['precision'], 'recall': metrics['recall'], 'f1': metrics['f1'], 'auroc': metrics['auroc'], 'auprc': metrics['auprc'], 'predicted_anomalies': metrics['predicted_anomalies'], 'best_sweep_f1': best_sweep['f1']})
score_sweep_df = pd.DataFrame(score_sweep_rows).sort_values(['f1', 'auprc', 'auroc'], ascending=False).reset_index(drop=True)
score_sweep_path = evaluation_dir / 'score_sweep_summary.csv'
score_sweep_df.to_csv(score_sweep_path, index=False)
display(score_sweep_df.head(15))
print(f'Saved score sweep summary to {score_sweep_path}')


In [ ]:
default_row = pd.DataFrame([{'name': 'default_config_score', 'precision': default_eval_summary['metrics_at_validation_threshold']['precision'], 'recall': default_eval_summary['metrics_at_validation_threshold']['recall'], 'f1': default_eval_summary['metrics_at_validation_threshold']['f1'], 'auroc': default_eval_summary['metrics_at_validation_threshold']['auroc'], 'auprc': default_eval_summary['metrics_at_validation_threshold']['auprc'], 'best_sweep_f1': default_eval_summary['best_threshold_sweep']['f1']}])
selected_row = score_sweep_df.iloc[0].copy()
selected_row_df = pd.DataFrame([selected_row])
comparison_df = pd.concat([default_row, selected_row_df[['name', 'precision', 'recall', 'f1', 'auroc', 'auprc', 'best_sweep_f1']]], ignore_index=True)
display(comparison_df)
selected_variant_summary = selected_row.to_dict()
selected_variant_summary_path = evaluation_dir / 'selected_score_variant.json'
selected_variant_summary_path.write_text(json.dumps(selected_variant_summary, indent=2), encoding='utf-8')
selected_variant_summary


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
repo_root = REPO_ROOT if 'REPO_ROOT' in globals() else Path.cwd()
if 'evaluation_dir' in globals():
    local_evaluation_dir = Path(evaluation_dir)
else:
    candidate_dirs = [repo_root / 'experiments' / 'anomaly_detection' / 'teacher_student' / 'resnet18' / 'x64' / 'main' / 'artifacts' / 'ts_resnet18' / 'results' / 'evaluation']
    existing_dirs = [path for path in candidate_dirs if path.exists()]
    if not existing_dirs:
        raise FileNotFoundError('Could not find a TS-Res18 evaluation directory. Ensure artifacts are present.')
    local_evaluation_dir = existing_dirs[0]
required_files = ['summary.json', 'selected_score_variant.json', 'val_scores.csv', 'test_scores.csv', 'threshold_sweep.csv']
missing_files = [f for f in required_files if not (local_evaluation_dir / f).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing evaluation artifacts: {missing_files}.')
default_summary_path = local_evaluation_dir / 'summary.json'
selected_variant_path = local_evaluation_dir / 'selected_score_variant.json'
default_eval_summary = json.loads(default_summary_path.read_text(encoding='utf-8'))
selected_variant = json.loads(selected_variant_path.read_text(encoding='utf-8'))
default_row = pd.DataFrame([{'name': 'default_config_score', 'precision': default_eval_summary['metrics_at_validation_threshold']['precision'], 'recall': default_eval_summary['metrics_at_validation_threshold']['recall'], 'f1': default_eval_summary['metrics_at_validation_threshold']['f1'], 'auroc': default_eval_summary['metrics_at_validation_threshold']['auroc'], 'auprc': default_eval_summary['metrics_at_validation_threshold']['auprc'], 'best_sweep_f1': default_eval_summary['best_threshold_sweep']['f1']}])
selected_row_df = pd.DataFrame([selected_variant])
comparison_df = pd.concat([default_row, selected_row_df[['name', 'precision', 'recall', 'f1', 'auroc', 'auprc', 'best_sweep_f1']]], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['default', 'best sweep variant'], comparison_df['f1'], color=['#8d99ae', '#2a9d8f'])
axes[0].set_title('Validation-Threshold F1')
axes[0].set_ylabel('F1')
axes[1].bar(['default', 'best sweep variant'], comparison_df['auroc'], color=['#8d99ae', '#2a9d8f'])
axes[1].set_title('AUROC')
axes[1].set_ylabel('AUROC')
score_variant_plot_path = plots_dir / 'score_variant_comparison.png'
plt.tight_layout()
plt.savefig(score_variant_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved score variant comparison plot to {score_variant_plot_path}')


## Defect-Type Breakdown

This cell recomputes the defect-type recall for the **selected deployed score** and saves it beside the evaluation outputs.


In [ ]:
metadata_path = REPO_ROOT / config['data']['metadata_csv']
test_metadata = pd.read_csv(metadata_path)
test_metadata = test_metadata[test_metadata['split'] == 'test'].reset_index(drop=True)
selected_scores_df = pd.read_csv(local_evaluation_dir / 'test_scores.csv').reset_index(drop=True)
if len(selected_scores_df) != len(test_metadata):
    raise ValueError(f'Length mismatch: scores={len(selected_scores_df)} metadata={len(test_metadata)}')
selected_threshold = float(selected_variant['threshold'])
analysis_df = test_metadata.copy()
analysis_df['score'] = selected_scores_df['score']
analysis_df['predicted_anomaly'] = (analysis_df['score'] > selected_threshold).astype(int)
defect_breakdown_df = analysis_df.loc[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean'), median_score=('score', 'median')).reset_index()
defect_breakdown_df['detected'] = defect_breakdown_df['detected'].astype(int)
defect_breakdown_df['missed'] = defect_breakdown_df['count'] - defect_breakdown_df['detected']
defect_breakdown_df['recall'] = defect_breakdown_df['detected'] / defect_breakdown_df['count']
defect_breakdown_df = defect_breakdown_df.sort_values(['recall', 'count', 'defect_type'], ascending=[True, False, True]).reset_index(drop=True)
display(defect_breakdown_df)
output_path = local_evaluation_dir / 'selected_defect_breakdown.csv'
defect_breakdown_df.to_csv(output_path, index=False)
print(f'Saved defect breakdown to {output_path}')


In [ ]:
plot_dir = plots_dir if 'plots_dir' in globals() else local_evaluation_dir.parent.parent / 'plots'
test_scores_df_display = pd.read_csv(local_evaluation_dir / 'test_scores.csv') if 'test_scores_df' not in locals() else test_scores_df
test_metadata_display = pd.read_csv(metadata_path)
test_metadata_display = test_metadata_display[test_metadata_display['split'] == 'test'].reset_index(drop=True)
test_display = test_metadata_display.copy()
test_display['score'] = test_scores_df_display['score'].values
threshold_for_display = default_eval_summary['metrics_at_validation_threshold']['threshold'] if 'default_eval_summary' in locals() else 0.0
top_n = 16
top_scored_idx = test_display.nlargest(top_n, 'score').index.tolist()
top_scored_data_display = test_display.loc[top_scored_idx].reset_index(drop=True)
image_size_display = int(config['data'].get('image_size', 64))
test_dataset_display = WaferMapDataset(metadata_path, split='test', image_size=image_size_display)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()
for idx, row_idx in enumerate(top_scored_idx):
    ax = axes[idx]
    image, label = test_dataset_display[row_idx]
    image_np = image.numpy()
    if image_np.shape[0] == 1:
        image_np = image_np[0]
    image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min() + 1e-06)
    ax.imshow(image_np, cmap='gray')
    score = top_scored_data_display.loc[idx, 'score']
    is_anomaly = top_scored_data_display.loc[idx, 'is_anomaly']
    defect_type = top_scored_data_display.loc[idx, 'defect_type'] if is_anomaly else 'normal'
    color = 'red' if is_anomaly else 'green'
    title = f'{defect_type}\n(score={score:.4f})'
    ax.set_title(title, color=color, fontsize=9, fontweight='bold')
    ax.axis('off')
plt.suptitle(f'Top {top_n} Highest-Scored Examples (Threshold={threshold_for_display:.4f})', fontsize=14, fontweight='bold')
plt.tight_layout()
top_scored_path = plot_dir / 'top_scored_examples.png'
plt.savefig(top_scored_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved top_scored_examples.png to {top_scored_path}')


## Saved Evaluation Plots

This section turns the saved validation/test score tables into the standard review plots for the selected deployed checkpoint. It does not retrain the model; it only reads the saved evaluation artifacts and writes figure files into `artifacts/ts_resnet18/plots/`.


In [ ]:
plot_dir = plots_dir if 'plots_dir' in globals() else local_evaluation_dir.parent.parent / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)
val_scores_df = pd.read_csv(local_evaluation_dir / 'val_scores.csv')
test_scores_df = pd.read_csv(local_evaluation_dir / 'test_scores.csv')
threshold_sweep_df = pd.read_csv(local_evaluation_dir / 'threshold_sweep.csv')
fig, ax = plt.subplots(figsize=(8, 4.5))
normal_scores = analysis_df.loc[analysis_df['is_anomaly'] == 0, 'score']
anomaly_scores = analysis_df.loc[analysis_df['is_anomaly'] == 1, 'score']
ax.hist(normal_scores, bins=40, alpha=0.65, label='normal', color='#457b9d')
ax.hist(anomaly_scores, bins=40, alpha=0.65, label='anomaly', color='#e76f51')
ax.axvline(selected_threshold, color='black', linestyle='--', label=f'threshold={selected_threshold:.4f}')
ax.set_title('Selected Score Distribution')
ax.set_xlabel('score')
ax.set_ylabel('count')
ax.legend()
score_distribution_path = plot_dir / 'score_distribution.png'
plt.tight_layout()
plt.savefig(score_distribution_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved score distribution plot to {score_distribution_path}')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision', color='#457b9d')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall', color='#f4a261')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1', color='#2a9d8f')
ax.axvline(selected_threshold, color='black', linestyle='--', label='selected threshold')
ax.set_title('Threshold Sweep')
ax.set_xlabel('threshold')
ax.set_ylabel('metric')
ax.legend()
threshold_sweep_path = plot_dir / 'threshold_sweep.png'
plt.tight_layout()
plt.savefig(threshold_sweep_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved threshold sweep plot to {threshold_sweep_path}')
tp = int(((analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1)).sum())
fn = int(((analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0)).sum())
fp = int(((analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1)).sum())
tn = int(((analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 0)).sum())
conf_matrix = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(conf_matrix, cmap='Blues')
for (row_idx, col_idx), value in np.ndenumerate(conf_matrix):
    ax.text(col_idx, row_idx, f'{value}', ha='center', va='center', color='black')
ax.set_xticks([0, 1], ['pred normal', 'pred anomaly'])
ax.set_yticks([0, 1], ['true normal', 'true anomaly'])
ax.set_title('Confusion Matrix')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
confusion_matrix_path = plot_dir / 'confusion_matrix.png'
plt.tight_layout()
plt.savefig(confusion_matrix_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved confusion matrix plot to {confusion_matrix_path}')
fig, ax = plt.subplots(figsize=(9, 4.5))
sorted_breakdown = defect_breakdown_df.sort_values(['recall', 'count', 'defect_type'], ascending=[True, False, True])
ax.barh(sorted_breakdown['defect_type'], sorted_breakdown['recall'], color='#2a9d8f')
ax.set_xlim(0, 1.0)
ax.set_xlabel('recall')
ax.set_title('Defect-Type Recall')
defect_breakdown_plot_path = plot_dir / 'defect_breakdown.png'
plt.tight_layout()
plt.savefig(defect_breakdown_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved defect breakdown plot to {defect_breakdown_plot_path}')
